In [ ]:
!pip install scikeras==0.13.0 scikit-learn==1.4.2

In [ ]:
#Librerías
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from matplotlib.gridspec import GridSpec

from sklearn.base import clone
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV
)

from sklearn.metrics import (
    classification_report,
    make_scorer,
    accuracy_score,
    f1_score,
    roc_auc_score,
    recall_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_curve,
    auc,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from scikeras.wrappers import KerasClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam, SGD


#Montamos Google Drive en Colab
from google.colab import drive
drive.mount('/content/drive')

#Ruta base de trabajo
RUTA_BASE = "/content/drive/MyDrive/NHANES/"

Cargamos las particiones de entrenamiento y test previamente preprocesadas y balanceadas mediante distintas estrategias

In [ ]:
# Carga datasets preprocesados y balanceados
x_train_desbal = pd.read_csv(RUTA_BASE + "x_train_desbal.csv")
x_train_UNDER  = pd.read_csv(RUTA_BASE + "x_train_UNDER.csv")
x_test_final   = pd.read_csv(RUTA_BASE + "x_test_final.csv")

# Convertimos las etiquetas categóricas ("Sí"/"No") a codificación binaria
# para entrenamiento supervisado.
def load_y(path):
    return (
        pd.read_csv(path)
        .squeeze()
        .map({"No": 0, "Sí": 1})
        .astype(int)
    )

y_train_desbal = load_y(RUTA_BASE + "y_train_desbal.csv")
y_train_UNDER  = load_y(RUTA_BASE + "y_train_UNDER.csv")
y_test_final   = load_y(RUTA_BASE + "y_test_final.csv")


# Calculamos el peso relativo de la clase minoritaria
scale = (y_train_desbal == 0).sum() / (y_train_desbal == 1).sum()
print(f"scale_pos_weight: {scale:.2f}")

# Copias explícitas para el entrenamiento de NN
x_under_df  = x_train_UNDER.copy()
x_desbal_df = x_train_desbal.copy()
x_test_df   = x_test_final.copy()

input_dim = x_under_df.shape[1]

# NN — codificación 0/1
y_train_under_nn  = pd.Series((y_train_UNDER  == 1).astype(int).values, name="stroke")
y_train_desbal_nn = pd.Series((y_train_desbal == 1).astype(int).values, name="stroke")
y_test_nn = pd.Series((y_test_final   == 1).astype(int).values, name="stroke")

Definición global de CV y métricas

In [ ]:
# Validación cruzada estratificada:
# mantiene la proporción de clases en cada fold
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=24
)

# Métricas utilizadas durante validación cruzada
scoring = {
    "Recall": make_scorer(recall_score, pos_label=1),
    "F1": make_scorer(f1_score, pos_label=1),
    "AUROC": "roc_auc",
    "Accuracy": "accuracy",
    "BalancedAccuracy": "balanced_accuracy"
}

metricas = [
    "mean_test_Accuracy",
    "mean_test_F1",
    "mean_test_AUROC",
    "mean_test_Recall",
    "mean_test_BalancedAccuracy"
]

# Entrenamiento de modelos

Creamos una funció general para entrenamiento y evaluación de modelos mediante RandomizedSearchCV

In [ ]:
# Función para entrenamiento por RandomizedSearchCV
def ejecutar_random_search(
    modelo,
    parametros,
    x_train,
    y_train,
    nombre_modelo,
    cv,
    scoring,
    metricas,
    n_iter=10,
    refit="AUROC",
    random_state=24,
    fit_params=None
):

    print("=" * 70)
    print(f"Entrenamiento: {nombre_modelo}")
    print("=" * 70)

    #Deteccion de red neuronal para variar n_jobs
    es_red_neuronal = (
    "keras" in str(type(modelo)).lower()   # detección por nombre de clase
    or isinstance(modelo, tf.keras.Model)  # detección directa de TF/Keras
    )

    # Asignación automática de n_jobs según tipo de modelo
    n_jobs = 1 if es_red_neuronal else -1

    # Configuración de RandomizedSearchCV
    random_search = RandomizedSearchCV(
        estimator=modelo,
        param_distributions=parametros,
        n_iter=n_iter,
        cv=cv,
        scoring=scoring,
        refit=refit,
        n_jobs=n_jobs,
        verbose=2,
        random_state=random_state
    )

    # Entrenamiento
    if fit_params is None:
        random_search.fit(x_train, y_train)
    else:
        random_search.fit(x_train, y_train, **fit_params)

    # Mejor modelo
    best_model = random_search.best_estimator_

    # Índice y resultados CV
    best_index = random_search.best_index_
    cv_results = random_search.cv_results_

    # Métricas medias del mejor modelo
    print("\nMétricas medias en validación cruzada:")

    for m in metricas:
        print(f"{m}: {cv_results[m][best_index]:.4f}")

    # Hiperparámetros del mejor modelo
    print("\nMejores hiperparámetros:")

    for clave, valor in random_search.best_params_.items():
        print(f"{clave}: {valor}")

    print("\n")

    # Return de la función
    return {
        "search": random_search,
        "best_model": best_model,
        "best_params": random_search.best_params_,
        "cv_results": cv_results,
        "best_index": best_index
    }

##Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Definimos el modelo base
rf = RandomForestClassifier(random_state=24)

### Undersampling

In [ ]:
# Rejilla inicial hiperparámetros
param_random_under1 = {
    "n_estimators":     [200, 400, 600],
    "max_depth":        [10, 20, 30],
    "min_samples_split":[2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features":     ["sqrt", "log2"]
}

# Entrenamiento Stage 1
resultado_rf_under1 = ejecutar_random_search(
    modelo=rf,
    parametros=param_random_under1,
    x_train=x_train_UNDER,
    y_train=y_train_UNDER,
    nombre_modelo="Random Forest + Undersampling (Stage 1)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=50
)

# Mostramos los resultados obtenidos
best_rf_under1 = resultado_rf_under1["best_model"]

Ajustamos la rejilla de hiperparámetros en función de los resultados obtenidos

In [ ]:
# Segunda rejilla de hiperparámetros
param_random_under2 = {
    "n_estimators":      [500, 600, 700],
    "max_depth":         [8, 10, 12],
    "min_samples_split": [2, 3, 4],
    "min_samples_leaf":  [1, 2, 3],
    "max_features":      ["log2", "sqrt"]
}

# Entrenamiento Stage 2
resultado_rf_under2 = ejecutar_random_search(
    modelo=rf,
    parametros=param_random_under2,
    x_train=x_train_UNDER,
    y_train=y_train_UNDER,
    nombre_modelo="Random Forest + Undersampling (Stage 2)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=10
)

# Mejor modelo final
best_rf_under2 = resultado_rf_under2["best_model"]

### Class_weight

In [ ]:
# Modelo con class_weight="balanced"
rf_cw = RandomForestClassifier(
    class_weight="balanced", # penalizamos errores en stroke
    random_state=24
)

In [ ]:
# Rejilla inicial de hiperparámetros
param_random_cw1 = {
    "n_estimators":      [200, 400, 600],
    "max_depth":         [10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf":  [1, 2, 4],
    "max_features":      ["sqrt", "log2"]
}

# Entrenamiento Stage 1
resultado_rf_cw1 = ejecutar_random_search(
    modelo=rf_cw,
    parametros=param_random_cw1,
    x_train=x_train_desbal,
    y_train=y_train_desbal,
    nombre_modelo="Random Forest + Class Weights (Stage 1)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=50
)

# Mostramos los resultados obtenidos
best_rf_cw1 = resultado_rf_cw1["best_model"]

Ajustamos la rejilla de hiperparámetros en función de los resultados obtenidos.

In [ ]:
# Segundo grid de hiperparámetros
param_random_cw2 = {
    "n_estimators":      [500, 600, 700],
    "max_depth":         [25, 30, 35],
    "min_samples_split": [4, 5, 6],
    "min_samples_leaf":  [3, 4, 5],
    "max_features":      ["log2", "sqrt"]
}

# Entrenamiento Stage 2
resultado_rf_cw2 = ejecutar_random_search(
    modelo=rf_cw,
    parametros=param_random_cw2,
    x_train=x_train_desbal,
    y_train=y_train_desbal,
    nombre_modelo="Random Forest + Class Weights (Stage 2)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=10
)

# Mejor modelo final
best_rf_cw2 = resultado_rf_cw2["best_model"]

### Undersampling + class_weight

In [ ]:
# Rejilla inicial de hiperparámetros
param_random_under_cw1 = {
    "n_estimators":      [200, 400, 600],
    "max_depth":         [10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf":  [1, 2, 4],
    "max_features":      ["sqrt", "log2"]
}

# Entrenamiento Stage 1
resultado_rf_under_cw1 = ejecutar_random_search(
    modelo=rf_cw,
    parametros=param_random_under_cw1,
    x_train=x_train_UNDER,
    y_train=y_train_UNDER,
    nombre_modelo="Random Forest + Class Weights + Undersampling (Stage 1)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=50
)

# Mostramos los resultados obtenidos
best_rf_under_cw1 = resultado_rf_under_cw1["best_model"]

Ajustamos la rejilla de hiperparámetros en función de los resultados obtenidos

In [ ]:
# Segunda rejilla de hiperparámetros
param_random_under_cw2 = {
    "n_estimators":      [500, 600, 700],
    "max_depth":         [15, 20, 25],
    "min_samples_split": [2, 3, 4],
    "min_samples_leaf":  [3, 4, 5],
    "max_features":      ["log2", "sqrt"]
}

# Entrenamiento Stage 2
resultado_rf_under_cw2 = ejecutar_random_search(
    modelo=rf_cw,
    parametros=param_random_under_cw2,
    x_train=x_train_UNDER,
    y_train=y_train_UNDER,
    nombre_modelo="Random Forest + Class Weights + Undersampling (Stage 2)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=50
)

# Mejor modelo final
best_rf_under_cw2 = resultado_rf_under_cw2["best_model"]

## XGBoost

In [ ]:
from xgboost import XGBClassifier

# Definimos el modelo base
xgb = XGBClassifier(random_state=24, eval_metric="auc")

### Undersampling

In [ ]:
# Rejilla inicial de hiperparámetros
param_xgb_under1 = {
    "n_estimators":    [100, 300, 500],
    "learning_rate":   [0.01, 0.1, 0.3],
    "max_depth":       [3, 6, 9],
    "subsample":       [0.6, 0.8, 1.0],
    "colsample_bytree":[0.6, 0.8, 1.0]
}

# Entrenamiento Stage 1
resultado_xgb_under1 = ejecutar_random_search(
    modelo=xgb,
    parametros=param_xgb_under1,
    x_train=x_train_UNDER,
    y_train=y_train_UNDER,
    nombre_modelo="XGBoost + Undersampling (Stage 1)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=50
)

# Mostramos los resultados obternidos
best_xgb_under1 = resultado_xgb_under1["best_model"]

Ajustamos la rejilla de hiperparámetros en función de los resultados obtenidos

In [ ]:
# Segunda rejilla de hiperparámetros
param_xgb_under2 = {
    "n_estimators":        [80, 100, 120],
    "max_depth":           [2, 3, 4],
    "learning_rate":       [0.05, 0.1, 0.15],
    "subsample":           [0.8, 1.0],
    "colsample_bytree":    [0.5, 0.6, 0.7]
}

# Entrenamiento Stage 2
resultado_xgb_under2 = ejecutar_random_search(
    modelo=xgb,
    parametros=param_xgb_under2,
    x_train=x_train_UNDER,
    y_train=y_train_UNDER,
    nombre_modelo="XGBoost + Undersampling (Stage 2)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=10
)

# Mejor modeo final
best_xgb_under2 = resultado_xgb_under2["best_model"]

### Scale_pos_weight

In [ ]:
# Definimos el modelo base con scale_pos_weight
xgb_cw = XGBClassifier(
    scale_pos_weight=scale,
    random_state=24,
    eval_metric="auc"
)

In [ ]:
# Rejilla inicial de hiperparámetros
param_xgb_cw1 = {
    "n_estimators":     [100, 300, 500],
    "learning_rate":    [0.01, 0.1, 0.3],
    "max_depth":        [3, 6, 9],
    "subsample":        [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0]
}

# Entrenamiento Stage 1
resultado_xgb_cw1 = ejecutar_random_search(
    modelo=xgb_cw,
    parametros=param_xgb_cw1,
    x_train=x_train_desbal,
    y_train=y_train_desbal,
    nombre_modelo="XGBoost + Class Weights (Stage 1)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=50
)

# Mostramos los resultados obtenidos
best_xgb_cw1 = resultado_xgb_cw1["best_model"]

Ajustamos la rejilla de hiperparámetros en función de los resultados obtenidos

In [ ]:
# Segunda rejilla de hiperparámetros
param_xgb_cw2 = {
    "n_estimators":        [250, 300, 350],
    "max_depth":           [2, 3, 4],
    "learning_rate":       [0.005, 0.01, 0.02],
    "subsample":           [0.5, 0.6, 0.7],
    "colsample_bytree":    [0.7, 0.8, 0.9]
}

# Entrenamiento Stage 2
resultado_xgb_cw2 = ejecutar_random_search(
    modelo=xgb_cw,
    parametros=param_xgb_cw2,
    x_train=x_train_desbal,
    y_train=y_train_desbal,
    nombre_modelo="XGBoost + Class Weights (Stage 2)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=10
)

# Mejor modelo final
best_xgb_cw2 = resultado_xgb_cw2["best_model"]


### Undersampling + scale_pos_weight

In [ ]:
# Rejilla inicial de hiperparámetros
param_xgb_under_cw1 = {
    "n_estimators":     [100, 300, 500],
    "learning_rate":    [0.01, 0.1, 0.3],
    "max_depth":        [3, 6, 9],
    "subsample":        [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0]
}

# Entrenamiento Stage 1
resultado_xgb_under_cw1 = ejecutar_random_search(
    modelo=xgb_cw,
    parametros=param_xgb_under_cw1,
    x_train=x_train_UNDER,
    y_train=y_train_UNDER,
    nombre_modelo="XGBoost + Undersampling + scale_pos_weight (Stage 1)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=50
)

# Mostramos los resultados obtenidos
best_xgb_under_cw1 = resultado_xgb_under_cw1["best_model"]

Ajustamos la rejilla de hiperparámetros en función de los resultados obtenidos

In [ ]:
# Segunda rejilla de hiperparámetros
param_xgb_under_cw2 = {
    "n_estimators":        [400, 500, 600],
    "max_depth":           [7, 9, 11],
    "learning_rate":       [0.005, 0.01, 0.02],
    "subsample":           [0.8, 1.0],
    "colsample_bytree":    [0.5, 0.6, 0.7]
}

# Entrenamiento Stage 2
resultado_xgb_under_cw2 = ejecutar_random_search(
    modelo=xgb_cw,
    parametros=param_xgb_under_cw2,
    x_train=x_train_UNDER,
    y_train=y_train_UNDER,
    nombre_modelo="XGBoost + Undersampling + scale_pos_weight (Stage 2)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=10
)

# Moejor modelo final
best_xgb_under_cw2 = resultado_xgb_under_cw2["best_model"]

##SVM

In [ ]:
from sklearn.svm import SVC

# Definimos el modelo base
svm = SVC(random_state=24, probability=True)

### Undersampling

In [ ]:
# Primera rejilla
param_svm_under1 = {
    "C":      [0.1, 1, 10, 100],
    "gamma":  ["scale", "auto", 0.001, 0.01, 0.1],
    "kernel": ["rbf", "linear"]
}

# Entrenamiento Stage 1
resultado_svm_under1 = ejecutar_random_search(
    modelo=svm,
    parametros=param_svm_under1,
    x_train=x_train_UNDER,
    y_train=y_train_UNDER,
    nombre_modelo="SVM + Undersampling (Stage 1)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=40
)

# Mostramos los resultados obtenidos
best_svm_under1 = resultado_svm_under1["best_model"]

Ajustamos la rejilla de hiperparámetros en función de los resultados obtenidos

In [ ]:
param_svm_under2 = {
     "C":     [5, 10, 15],
    "gamma":  [0.005, 0.01, 0.02],
    "kernel": ["rbf"]
}

# Entrenamiento Stage 2
resultado_svm_under2 = ejecutar_random_search(
    modelo=svm,
    parametros=param_svm_under2,
    x_train=x_train_UNDER,
    y_train=y_train_UNDER,
    nombre_modelo="SVM + Undersampling (Stage 2)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=10
)

# Mejor modelo final
best_svm_under2 = resultado_svm_under2["best_model"]

### Class_weight

In [ ]:
# Definimos el modelo base
svm_cw = SVC(random_state=24, probability=True, class_weight="balanced")

In [ ]:
# Rejilla inicial de hiperparámetros
param_svm_cw1 = {
    "C":      [0.1, 1, 10, 100],
    "gamma":  ["scale", "auto", 0.001, 0.01, 0.1],
    "kernel": ["rbf", "linear"]
}

# Entrenamiento Stage 1
resultado_svm_cw1 = ejecutar_random_search(
    modelo=svm_cw,
    parametros=param_svm_cw1,
    x_train=x_train_desbal,
    y_train=y_train_desbal,
    nombre_modelo="SVM + Class Weights (Stage 1)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=10
)

# Mostramos los resultados obtenidos
best_svm_cw1 = resultado_svm_cw1["best_model"]

Ajustamos la rejilla de hiperparámetros en función de los resultados obtenidos

In [ ]:
# Segunda rejilla de hiperparámetros
param_svm_cw2 = {
    "C":      [0.5, 1, 2],
    "gamma":  [0.005, 0.01, 0.02],
    "kernel": ["rbf"]
}

# Entrenamiento Stage 2
resultado_svm_cw2 = ejecutar_random_search(
    modelo=svm_cw,
    parametros=param_svm_cw2,
    x_train=x_train_desbal,
    y_train=y_train_desbal,
    nombre_modelo="SVM + Class Weights (Stage 2)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=10
)

# Mejor modelo final
best_svm_cw2 = resultado_svm_cw2["best_model"]

### Undersampling + class_weights

In [ ]:
# Rejilla inicial de hiperparámetros
param_svm_under_cw1 = {
    "C":      [0.1, 1, 10, 100],
    "gamma":  ["scale", "auto", 0.001, 0.01, 0.1],
    "kernel": ["rbf", "linear"]
}

# Entrenamiento Stage 1
resultado_svm_under_cw1 = ejecutar_random_search(
    modelo=svm_cw,
    parametros=param_svm_under_cw1,
    x_train=x_train_UNDER,
    y_train=y_train_UNDER,
    nombre_modelo="SVM + Undersampling + Class Weights (Stage 1)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=10
)

# Mostramos el resultado obtenido
best_svm_under_cw1 = resultado_svm_under_cw1["best_model"]

Ajustamos la rejilla de hiperparámetros en función de los resultados obtenidos

In [ ]:
# Segunda rejilla de hiperparámetros
param_svm_under_cw2 = {
    "C":      [5, 10, 20],
    "gamma":  [0.005, 0.01, 0.02],
    "kernel": ["rbf"]
}

# Entrenamiento Stage 2
resultado_svm_under_cw2 = ejecutar_random_search(
    modelo=svm_cw,
    parametros=param_svm_under_cw2,
    x_train=x_train_UNDER,
    y_train=y_train_UNDER,
    nombre_modelo="SVM + Undersampling + Class Weights (Stage 1)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=10
)

# Mejor modelo final
best_svm_under_cw2 = resultado_svm_under_cw2["best_model"]

## Regresión Logística

In [ ]:
from sklearn.linear_model import LogisticRegression

# Definimos el modelo base
lr = LogisticRegression(random_state=24, max_iter=1000)

# Evitamos warnings en las combinaciones incorrectas entre penalty y solver
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

### Undersampling

In [ ]:
# Rejilla inicial de hiperparámetros
param_lr_under1 = {
    "C":       [0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver":  ["liblinear", "saga", "lbfgs"]
}

# Entrenamiento Stage 1
resultado_lr_under1 = ejecutar_random_search(
    modelo=lr,
    parametros=param_lr_under1,
    x_train=x_train_UNDER,
    y_train=y_train_UNDER,
    nombre_modelo="Logistic Regression + Undersampling (Stage 1)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=50
)
# Mostramos los resultados obtenidos
best_lr_under1 = resultado_lr_under1["best_model"]

Ajustamos la rejilla de hiperparámetros en función de los resultados obtenidos

In [ ]:
# Segunda rejilla de hiperparámetros
param_lr_under2 = {
    "C":       [0.5, 1, 2, 5],
    "penalty": ["l1"],
    "solver":  ["saga"]
}

# Entrenamiento Stage 2
resultado_lr_under2 = ejecutar_random_search(
    modelo=lr,
    parametros=param_lr_under2,
    x_train=x_train_UNDER,
    y_train=y_train_UNDER,
    nombre_modelo="Logistic Regression + Undersampling (Stage 2)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=5
)
# Mejor modelo final
best_lr_under2 = resultado_lr_under2["best_model"]

### Class_weight

In [ ]:
# Definimos el modelo base
lr_cw = LogisticRegression(random_state=24, max_iter=1000, class_weight="balanced")

In [ ]:
# Rejilla inicial de hiperparámetros
param_lr_cw1 = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear", "saga", "lbfgs"]
}

# Entrenamiento Stage 1
resultado_lr_cw1 = ejecutar_random_search(
    modelo=lr_cw,
    parametros=param_lr_cw1,
    x_train=x_train_desbal,
    y_train=y_train_desbal,
    nombre_modelo="Logistic Regression + Class Weight (Stage 1)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=50
)

# Mostramos los resultados obtenidos
best_lr_cw1 = resultado_lr_cw1["best_model"]

Ajustamos la rejilla de hiperparámetros en función de los resultados obtenidos

In [ ]:
param_lr_cw2 = {
    "C":       [0.05, 0.1, 0.2, 0.5, 1],
    "penalty": ["l1"],
    "solver":  ["liblinear"]
}

# Entrenamiento Stage 2
resultado_lr_cw2 = ejecutar_random_search(
    modelo=lr_cw,
    parametros=param_lr_cw2,
    x_train=x_train_desbal,
    y_train=y_train_desbal,
    nombre_modelo="Logistic Regression + Class Weight (Stage 2)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=5
)

# Mejor modelo final
best_lr_cw2 = resultado_lr_cw2["best_model"]

### Undersampling + class_weights

In [ ]:
# Rejilla inicial de hiperparámetros
param_lr_under_cw1 = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear", "saga", "lbfgs"]
}

# Entrenamiento Stage 1
resultado_lr_under_cw1 = ejecutar_random_search(
    modelo=lr_cw,
    parametros=param_lr_under_cw1,
    x_train=x_train_UNDER,
    y_train=y_train_UNDER,
    nombre_modelo="Logistic Regression + Under + CW (Stage 1)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=50
)

# Mostramos los resultados obtenidos
best_lr_under_cw1 = resultado_lr_under_cw1["best_model"]

Ajustamos la rejilla de hiperparámetros en función de los resultados obtenidos

In [ ]:
# Segunda rejilla de hiperparámetros
param_lr_under_cw2 = {
    "C":       [0.5, 1, 2, 5, 10],
    "penalty": ["l1"],
    "solver":  ["saga"]
}

# Entrenamiento Stage 2
resultado_lr_under_cw2 = ejecutar_random_search(
    modelo=lr_cw,
    parametros=param_lr_under_cw2,
    x_train=x_train_UNDER,
    y_train=y_train_UNDER,
    nombre_modelo="Logistic Regression + Under + CW (Stage 2)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=5
)

# Mejor modelo final
best_lr_under_cw2 = resultado_lr_under_cw2["best_model"]

In [ ]:
# ── Sklearn y XGBoost ────────────────────────────────────────
modelos_finales = {
    "rf_under":  best_rf_under2,
    "xgb_under": best_xgb_under2,
    "svm_under": best_svm_under2,
    "lr_under":  best_lr_under2,
}

with open(RUTA_BASE + "modelos_finales.pkl", "wb") as f:
    pickle.dump(modelos_finales, f)

## Redes neuronales

Generamos la función para construir el modelo

In [ ]:
# Guardamos la dimensión de entrada (número de variables predictoras)
input_dim = x_under_df.shape[1]

# Función para construir el modelo
def build_model(hidden_layer_1, dropout_1, optimizer_name, learning_rate):

    # Definimos la arquitectura secuencial de la red
    model = keras.Sequential([

        # Capa oculta: neuronas con activación ReLU
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(hidden_layer_1, activation="relu"),

        # Dropout: durante el entrenamiento apaga aleatoriamente un % de neuronas
        keras.layers.Dropout(dropout_1),

        # Capa de salida: una sola neurona con activación sigmoid
        keras.layers.Dense(1, activation="sigmoid")
    ])

    # Seleccionamos el optimizador según el parámetro recibido
    if optimizer_name == "adam":
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == "sgd":
        optimizer = keras.optimizers.SGD(learning_rate=learning_rate)

    # Compilamos el modelo
    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy", #función de pérdida estándar para clasificación binaria
        metrics=["AUC"] #métrica de evaluación durante el entrenamiento
    )

    return model

#Definimos el wrapper de Scikeras
mlp = KerasClassifier(
    model=build_model,
    epochs=100,
    batch_size=32,
    verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor = "loss",
        patience=10,
        restore_best_weights=True
    )]
)


### Undersampling

In [ ]:
# Rejilla inicial de hiperparámetros
param_grid_nn_under1 = {
    "model__hidden_layer_1": [32, 64, 128],
    "model__dropout_1":      [0.2, 0.3, 0.5],
    "model__optimizer_name": ["adam", "sgd"],
    "model__learning_rate":  [0.001, 0.01]
}

# Entrenamiento Stage 1
resultado_nn_under1 = ejecutar_random_search(
    modelo=mlp,
    parametros=param_grid_nn_under1,
    x_train=x_under_df,
    y_train=y_train_under_nn,
    nombre_modelo="NN + Undersampling (Stage 1)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=36,

)

# Mostramos los resultados obtenidos
best_nn_under1 = resultado_nn_under1["best_model"]

Ajustamos la rejilla de hiperparámetros en función de los resultados obtenidos

In [ ]:
# Segunda rejilla de hiperparámetros
param_grid_nn_under2 = {
    "model__optimizer_name": ["sgd", "adam"],
    "model__learning_rate":  [0.005, 0.01, 0.02],
    "model__hidden_layer_1": [64, 128, 256],
    "model__dropout_1":      [0.1, 0.2, 0.3]
}

# Entrenamiento Stage 2
resultado_nn_under2 = ejecutar_random_search(
    modelo=mlp,
    parametros=param_grid_nn_under2,
    x_train=x_under_df,
    y_train=y_train_under_nn,
    nombre_modelo="NN + Undersampling (Stage 2)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=10
)

# Mejor modelo final
best_nn_under2 = resultado_nn_under2["best_model"]

### Class_weight

In [ ]:
# Rejilla inicial de hiperparámetros
param_grid_nn_cw1 = {
    "model__hidden_layer_1": [32, 64, 128],
    "model__dropout_1":      [0.2, 0.3, 0.5],
    "model__optimizer_name": ["adam", "sgd"],
    "model__learning_rate":  [0.001, 0.01]
}

# Entrenamiento Stage 1
resultado_nn_cw1 = ejecutar_random_search(
    modelo=mlp,
    parametros=param_grid_nn_cw1,
    x_train=x_desbal_df,
    y_train=y_train_desbal_nn,
    nombre_modelo="NN + Class Weight (Stage 1)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=36,
    fit_params={
        "class_weight": {0: 1, 1: scale}
    }
)

# Mostramos los resultados obtenidos
best_nn_cw1 = resultado_nn_cw1["best_model"]

Ajustamos la rejilla de hiperparámetros en función de los resultados obtenidos

In [ ]:
# Segunda rejilla de hiperparámetros
param_grid_nn_cw2 = {
    "model__optimizer_name": ["sgd"],
    "model__learning_rate":  [0.005, 0.01, 0.02],
    "model__hidden_layer_1": [16, 32, 64],
    "model__dropout_1":      [0.4, 0.5, 0.6]
}

# Entrenamiento Stage 2
resultado_nn_cw2 = ejecutar_random_search(
    modelo=mlp,
    parametros=param_grid_nn_cw2,
    x_train=x_desbal_df,
    y_train=y_train_desbal_nn,
    nombre_modelo="NN + Class Weight (Stage 2)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=10,
    fit_params={
        "class_weight": {0: 1, 1: scale}
    }
)

# Mejor modelo final
best_nn_cw2 = resultado_nn_cw2["best_model"]

### Undersampling + class_weight

In [ ]:
# Rejilla inicial de hiperparámetros
param_grid_nn_under_cw1 = {
    "model__hidden_layer_1": [32, 64, 128],
    "model__dropout_1":      [0.2, 0.3, 0.5],
    "model__optimizer_name": ["adam", "sgd"],
    "model__learning_rate":  [0.001, 0.01]
}

# Entrenamiento Stage 1
resultado_nn_under_cw1 = ejecutar_random_search(
    modelo=mlp,
    parametros=param_grid_nn_under_cw1,
    x_train=x_under_df,
    y_train=y_train_under_nn,
    nombre_modelo="NN + Under + Class Weight (Stage 1)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=10,
    fit_params={
        "class_weight": {0: 1, 1: scale}
    }
)

# Mostramos los resultados obtenidos
best_nn_under_cw1 = resultado_nn_under_cw1["best_model"]

Ajustamos la rejilla de hiperparámetros en función de los resultados obtenidos

In [ ]:
# Segunda rejilla de hiperparámetros
param_grid_nn_under_cw2 = {
    "model__optimizer_name": ["adam"],
    "model__learning_rate":  [0.0005, 0.001, 0.002],
    "model__hidden_layer_1": [16, 32, 64],
    "model__dropout_1":      [0.1, 0.2, 0.3]
}

# Entrenamiento Stage 2
resultado_nn_under_cw2 = ejecutar_random_search(
    modelo=mlp,
    parametros=param_grid_nn_under_cw2,
    x_train=x_under_df,
    y_train=y_train_under_nn,
    nombre_modelo="NN + Under + Class Weight (Stage 2)",
    cv=cv,
    scoring=scoring,
    metricas=metricas,
    n_iter=10,
    fit_params={
        "class_weight": {0: 1, 1: scale}
    }
)

# Mejor modelo final
best_nn_under_cw2 = resultado_nn_under_cw2["best_model"]

###

# Validación de modelos

Creamos una función de validación y visualización

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    recall_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_curve,
    auc,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Función de validación y visualización en test
def evaluar_modelo(
    modelo,
    x_test,
    y_test,
    nombre_modelo,
    umbral=0.5,
    es_nn=False
):

    # ── Predicciones ─────────────────────────────────────
    if es_nn:
        y_prob = modelo.predict_proba(x_test)[:, 1]
        y_pred = (y_prob >= umbral).astype(int)
    else:
        y_pred = modelo.predict(x_test)
        y_prob = modelo.predict_proba(x_test)[:, 1]

    # ── Métricas ─────────────────────────────────────────
    metricas = pd.DataFrame({
        "Test": [
            accuracy_score(y_test, y_pred),
            f1_score(y_test, y_pred, pos_label=1),
            roc_auc_score(y_test, y_prob),
            recall_score(y_test, y_pred, pos_label=1),
            balanced_accuracy_score(y_test, y_pred),
            average_precision_score(y_test, y_prob)
        ]
    }, index=[
        "Accuracy",
        "F1",
        "AUROC",
        "Recall",
        "Balanced Accuracy",
        "Average Precision"
    ])

    print(metricas.round(4))

    # ── Gráficos ─────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auroc = auc(fpr, tpr)

    axes[0].plot(
        fpr,
        tpr,
        lw=2,
        label=f'AUROC = {auroc:.4f}'
    )

    axes[0].plot([0, 1], [0, 1], linestyle='--', lw=1)

    axes[0].set_xlabel('Tasa de Falsos Positivos')
    axes[0].set_ylabel('Tasa de Verdaderos Positivos')
    axes[0].set_title(f'Curva ROC — {nombre_modelo}')
    axes[0].legend(loc='lower right')

    # Matriz de confusión
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No", "Sí"]
    ).plot(ax=axes[1], colorbar=False, cmap="Blues")

    axes[1].set_title(f'Matriz de Confusión — {nombre_modelo}')

    plt.tight_layout()
    plt.show()

    return metricas

## Random Forest

### Undersampling

In [ ]:
## Validación RF + Undersampling
metricas_rf_under = evaluar_modelo(
    best_rf_under2,
    x_test_final,
    y_test_final,
    "RF Undersampling"
)

### Class_weight

In [ ]:
# Validación RF + Class weights
metricas_rf_cw = evaluar_modelo(
    best_rf_cw2,
    x_test_final,
    y_test_final,
    "RF Class Weight"
)

### Undersampling + class_weight

In [ ]:
## Validación RF + Undersampling + Class weights
metricas_rf_under_cw = evaluar_modelo(
    best_rf_under_cw2,
    x_test_final,
    y_test_final,
    "RF Under + Class Weight"
)

## XGBoost

### Undersampling

In [ ]:
## Validación XGBoost + Undersampling
metricas_xgb_under = evaluar_modelo(
    best_xgb_under2,
    x_test_final,
    y_test_final,
    "XGBoost Undersampling"
)

### Scale_pos_weight

In [ ]:
## Validación XGBoost + scale_pos_weight
metricas_xgb_cw = evaluar_modelo(
    best_xgb_cw2,
    x_test_final,
    y_test_final,
    "XGBoost scale_pos_weight"
)

###Undersampling + scale_pos_weight

In [ ]:
## Validación XGBoost + Undersampling + scale_pos_weight
metricas_xgb_under_cw = evaluar_modelo(
    best_xgb_under_cw2,
    x_test_final,
    y_test_final,
    "XGBoost Under + scale_pos_weight"
)

## SVM

### Undersampling

In [ ]:
## Validación SVM + Undersampling
metricas_svm_under = evaluar_modelo(
    best_svm_under2,
    x_test_final,
    y_test_final,
    "SVM Undersampling"
)

### Class_weight

In [ ]:
## Validación SVM + Class Weight
metricas_svm_cw = evaluar_modelo(
    best_svm_cw2,
    x_test_final,
    y_test_final,
    "SVM Class Weight")

###Undersampling + class_weight

In [ ]:
## Validación SVM + Undersampling + Class Weight
metricas_svm_under_cw = evaluar_modelo(
    best_svm_under_cw2,
    x_test_final,
    y_test_final,
    "SVM Under + Class Weight"
)

## Regresión Logística

###Undersampling

In [ ]:
## Validación LR + Undersampling
metricas_lr_under = evaluar_modelo(
    best_lr_under2,
    x_test_final,
    y_test_final,
    "LR Undersampling"
)

### Class_weight

In [ ]:
### Validación LR + Class Weight
metricas_lr_cw = evaluar_modelo(
    best_lr_cw2,
    x_test_final,
    y_test_final,
    "LR Class Weight"
)

### Undersampling + class_weight

In [ ]:
## Validación LR + Undersampling + Class Weight
metricas_lr_under_cw = evaluar_modelo(
    best_lr_under_cw2,
    x_test_final,
    y_test_final,
    "LR Under + Class Weight"
)

## Redes neuronales

### Undersampling

In [ ]:
### Validación NN + Undersampling
metricas_nn_under = evaluar_modelo(
    best_nn_under2,
    x_test_df,
    y_test_nn,
    "NN Undersampling",
    es_nn=True
)

### Class_weight

In [ ]:
### Validación NN + Class Weight
metricas_nn_cw = evaluar_modelo(
    best_nn_cw2,
    x_test_df,
    y_test_nn,
    "NN Class Weight",
    es_nn=True
)

### Undersampling + class_weight

In [ ]:
### Validación NN + Undersampling + Class Weight

metricas_nn_under_cw = evaluar_modelo(
    best_nn_under_cw2,
    x_test_df,
    y_test_nn,
    "NN Under + Class Weight",
    es_nn=True
)

Se seleccionan los modelos entrenados con undersampling para la fase final de interpretabilidad, priorizando consistencia metodológica y estabilidad interpretativa

In [ ]:
# ── Sklearn y XGBoost ────────────────────────────────────────
modelos_finales = {
    "rf_under":  best_rf_under2,
    "xgb_under": best_xgb_under2,
    "svm_under": best_svm_under2,
    "lr_under":  best_lr_under2,
}

with open(RUTA_BASE + "modelos_finales.pkl", "wb") as f:
    pickle.dump(modelos_finales, f)

In [ ]:
# ── Red Neuronal ────────────────────────────────────────
model = best_nn_under2.model_
model.save(RUTA_BASE + "nn_under.keras")

# Gráfico entrenamiento

In [ ]:
RUTA_BASE = "/content/drive/MyDrive/NHANES/"
N_SPLITS, SEED = 5, 24   # nº folds de CV y semilla común

# Métricas usadas en la CV multimétrica
scoring = {
    "AUROC":   "roc_auc",
    "Recall":  make_scorer(recall_score, pos_label=1),
    "AvgPrec": "average_precision",
}

# Modelos sklearn/XGBoost guardados en pickle + red neuronal en formato Keras
with open(RUTA_BASE + "modelos_finales.pkl", "rb") as f:
    modelos = pickle.load(f)
modelos["nn_under"] = load_model(RUTA_BASE + "nn_under.keras")

# Particiones train (UNDER) y test. Convertimos las etiquetas "Sí"/"No" a 1/0
X_train = pd.read_csv(RUTA_BASE + "x_train_UNDER.csv")
y_train = (pd.read_csv(RUTA_BASE + "y_train_UNDER.csv").squeeze() == "Sí").astype(int)
X_test  = pd.read_csv(RUTA_BASE + "x_test_final.csv")
y_test  = (pd.read_csv(RUTA_BASE + "y_test_final.csv").squeeze() == "Sí").astype(int)

# Etiquetas legibles para los gráficos y tablas
LABELS = {"rf_under": "RF", "xgb_under": "XGB", "svm_under": "SVM",
          "lr_under": "LR", "nn_under": "NN"}

# Helper para obtener probabilidades de la clase positiva.
# Unifica el comportamiento de los modelos sklearn (predict_proba) y de la
# red neuronal Keras pura (predict + ravel)
def proba(modelo, X):
    if hasattr(modelo, "predict_proba"):
        return modelo.predict_proba(X)[:, 1]
    return modelo.predict(X.values if hasattr(X, "values") else X).ravel()

Validación cruzada multimétrica + boxplot

In [ ]:
# La NN se reentrena dentro de la CV (no se puede clonar un modelo Keras
# ya ajustado), así que la reconstruimos con los hiperparámetros finales
# Función crear_nn() adaptada a NHANES
def crear_nn():
    m = Sequential([
        Input(shape=(X_train.shape[1],)),
        Dense(128, activation="relu"),
        Dropout(0.1),
        Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=SGD(0.01),
              loss="binary_crossentropy", metrics=["AUC"])
    return m

# CV estratificada compartida por todos los modelos
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
records = []

# Cross-validate sobre cada modelo. Para la NN usamos un KerasClassifier
# nuevo (se entrena de cero en cada fold); para el resto, clone() basta.
for nombre, modelo in modelos.items():
    if nombre == "nn_under":
        estimator = KerasClassifier(model=crear_nn, epochs=100, batch_size=32,
                                    verbose=0, random_state=SEED)
        scores = cross_validate(estimator, X_train, y_train, cv=cv, scoring=scoring)
    else:
        scores = cross_validate(clone(modelo), X_train, y_train, cv=cv,
                                scoring=scoring, n_jobs=-1)
    # Aplanamos los scores a formato largo (un score por fila) para seaborn
    for met in scoring:
        for s in scores[f"test_{met}"]:
            records.append({"Modelo": LABELS[nombre], "Métrica": met, "Score": s})

# Convertimos a DataFrame y fijamos orden categórico para el gráfico
df_cv = pd.DataFrame(records)
for col, cats in {"Modelo":  ["RF", "XGB", "SVM", "LR", "NN"],
                  "Métrica": ["AUROC", "Recall", "AvgPrec"]}.items():
    df_cv[col] = pd.Categorical(df_cv[col], categories=cats, ordered=True)

# Boxplot con las tres métricas agrupadas por modelo
palette = {"AUROC": "#378ADD", "Recall": "#F0AD4E", "AvgPrec": "#0F6E56"}
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df_cv, x="Modelo", y="Score", hue="Métrica",
            palette=palette, width=0.7, whis=(0, 100), showfliers=False, ax=ax)
ax.set(xlabel="Modelo", ylabel="Valor de la métrica (CV 5-fold)",
       title="Rendimiento por modelo y métrica — NHANES",   # <-- NHANES
       ylim=(0.70, 0.90))
ax.legend(title="Métrica", frameon=False,
          bbox_to_anchor=(1.02, 0.5), loc="center left")  # leyenda fuera del plot
ax.grid(axis="y", alpha=0.3, linestyle="--")
ax.set_axisbelow(True)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.savefig(RUTA_BASE + "cv_boxplot_nhanes.png", dpi=150, bbox_inches="tight")
plt.show()

# Tabla resumen (media y desviación típica por modelo y métrica)
resumen = (df_cv.groupby(["Modelo", "Métrica"], observed=True)["Score"]
                .agg(["mean", "std"]).round(4))
df_cv.to_csv(RUTA_BASE + "cv_scores_nhanes.csv", index=False)
print(resumen)

Predicciones sobre test y tabla de métricas

In [ ]:
# Calculamos probabilidades y predicciones binarias (umbral 0.5) una sola vez
probs = {LABELS[n]: proba(m, X_test) for n, m in modelos.items()}
preds = {k: (p >= 0.5).astype(int) for k, p in probs.items()}

# Tabla de las tres métricas para cada modelo
metricas_test = pd.DataFrame({
    k: [roc_auc_score(y_test, probs[k]),
        average_precision_score(y_test, probs[k]),
        recall_score(y_test, preds[k], pos_label=1)]
    for k in probs
}, index=["AUROC", "Average Precision", "Recall"])
print(metricas_test.round(4))

Figura ROC con los cinco modelos

In [ ]:
orden = ["RF", "XGB", "SVM", "LR", "NN"]

# Layout compartido: 3 paneles arriba (cols 0-1, 2-3, 4-5) y 2 centrados abajo (1-2, 3-4)
def crear_axes_5paneles(fig):
    gs = GridSpec(2, 6, figure=fig, hspace=0.45, wspace=0.40)
    return [fig.add_subplot(gs[0, i:i+2]) for i in (0, 2, 4)] + \
           [fig.add_subplot(gs[1, i:i+2]) for i in (1, 3)]

# --- Figura ROC ---
fig = plt.figure(figsize=(16, 9))
for axis, nombre in zip(crear_axes_5paneles(fig), orden):
    fpr, tpr, _ = roc_curve(y_test, probs[nombre], pos_label=1)
    axis.plot(fpr, tpr, color="steelblue", lw=2, label=f"AUROC = {auc(fpr, tpr):.4f}")
    axis.plot([0, 1], [0, 1], color="gray", linestyle="--", lw=1)
    axis.set(xlabel="Tasa de falsos positivos", ylabel="Tasa de verdaderos positivos",
             title=nombre, xlim=(0, 1), ylim=(-0.02, 1.06))
    axis.legend(loc="lower right", fontsize=9)
    axis.grid(True, alpha=0.3)
fig.suptitle("Curvas ROC por modelo — NHANES (undersampling)", fontsize=13, y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(RUTA_BASE + "figura_4_3a_roc_nhanes.png", dpi=300, bbox_inches="tight")
plt.show()

# --- Figura matrices de confusión (mismo layout) ---
fig = plt.figure(figsize=(16, 9))
for axis, nombre in zip(crear_axes_5paneles(fig), orden):
    y_pred = (probs[nombre] >= 0.5).astype(int)
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    cm_norm = cm / cm.sum(axis=1, keepdims=True)  # normalizada por clase real (recall por fila)
    etiquetas = np.array([[f"{c}\n({p:.2f})" for c, p in zip(fc, fp)]
                          for fc, fp in zip(cm, cm_norm)])
    sns.heatmap(cm_norm, annot=etiquetas, fmt="", cmap="Blues", cbar=False,
                vmin=0, vmax=1, square=True, ax=axis, annot_kws={"fontsize": 9},
                xticklabels=["No ictus", "Ictus"], yticklabels=["No ictus", "Ictus"])
    axis.set(xlabel="Predicción", ylabel="Valor real", title=nombre)
fig.suptitle("Matrices de confusión por modelo — NHANES (undersampling, umbral = 0,5)",
             fontsize=13, y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(RUTA_BASE + "figura_4_3b_confusion_nhanes.png", dpi=300, bbox_inches="tight")
plt.show()